# Train LesionLens on HAM10000 (Kaggle or Colab GPU)

This notebook runs the repository's own training pipeline on a free GPU and produces a
model artifact you can drop into `models/` on your machine.

**Before running**

1. Kaggle: *Add data* -> search **"Skin Cancer MNIST: HAM10000"** (kmader) and attach it.
   Enable a **GPU** accelerator in the notebook settings.
2. Read and accept the dataset licence: HAM10000 is **CC BY-NC 4.0** (non-commercial,
   attribution to Tschandl et al., *Sci. Data* 5, 180161, 2018).
3. Set `REPO_URL` below to your copy of this repository.

Nothing here fabricates data or results: metrics come from the held-out, lesion-grouped
test split and are written to `metrics.json` next to the weights.

In [ ]:
REPO_URL = "https://github.com/<your-account>/lesionlens.git"  # <- change me
DATASET_DIR = "/kaggle/input/skin-cancer-mnist-ham10000"       # Kaggle path; adjust on Colab

!git clone --depth 1 $REPO_URL lesionlens
%cd lesionlens
!pip install -q -r ml/requirements.txt && pip install -q -e .

In [ ]:
import torch, os, glob
print("CUDA available:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
print(sorted(os.listdir(DATASET_DIR)))
image_dirs = sorted(d for d in glob.glob(f"{DATASET_DIR}/*") if os.path.isdir(d) and "images" in d.lower())
print(image_dirs)

## 1. Lesion-grouped split

Several HAM10000 images show the same lesion. Splitting by `lesion_id` keeps every image
of a lesion in one split, so test metrics are not inflated by near-duplicates.

In [ ]:
dirs = " ".join(image_dirs)
!python -m ml.datasets.prepare_ham10000 --metadata {DATASET_DIR}/HAM10000_metadata.csv --images {dirs} --output data/processed --mode symlink

## 2. Train

EfficientNet-B0 from ImageNet weights, class-weighted loss, warm-up + cosine schedule,
early stopping on validation macro-F1, temperature calibration, then evaluation on the
held-out test split. Override any setting with `--set key=value`.

In [ ]:
!python -m ml.training.train --config ml/configs/efficientnet_b0.yaml --set data.num_workers=2 --set data.batch_size=64

In [ ]:
import json
m = json.load(open("models/efficientnet_b0-v1.0.0/metrics.json"))["metrics"]
print({k: m[k] for k in ("n_samples", "accuracy", "balanced_accuracy", "roc_auc_macro_ovr")}, "macro-F1:", m["macro"]["f1"])
for row in m["per_class"]:
    print(f"{row['code']:>6}  support={row['support']:>4}  recall={row['recall']}  specificity={row['specificity']}  auc={row['roc_auc']}")

## 3. Download the artifact

Unzip it into `models/` in your local repository and set
`MODEL_PATH=models/efficientnet_b0-v1.0.0` (or `MODEL_DIR=efficientnet_b0-v1.0.0` for Docker).
The `checkpoints/` folder is only needed to resume training and is excluded.

In [ ]:
!cd models && zip -qr ../efficientnet_b0-v1.0.0.zip efficientnet_b0-v1.0.0 -x "*/checkpoints/*" && cd .. && ls -lh efficientnet_b0-v1.0.0.zip